In [1]:
import os
import glob
import torch
import numpy as np
from astropy.table import Table as aT
from astropy.cosmology import Planck13

In [2]:
lrgs = aT.read('/global/cfs/projectdirs/desi/survey/catalogs/Y1/LSS/iron/LSScats/v1.2/LRG_full.dat.fits')

In [3]:
has_redshift = ~lrgs['Z_not4clus'].mask

In [4]:
print(np.sum(has_redshift), np.mean(has_redshift))

2477439 0.691844854264983


In [5]:
lrgs_chunk = aT()
lrgs_chunk['TARGETID'] = lrgs['TARGETID'][has_redshift]
lrgs_chunk['RA'] = lrgs['RA'][has_redshift]
lrgs_chunk['DEC'] = lrgs['DEC'][has_redshift]
lrgs_chunk['Z_not4clus'] = lrgs['Z_not4clus'][has_redshift]

lrgs_chunk['FLUX_G_CORR'] = lrgs['FLUX_G'][has_redshift] / lrgs['MW_TRANSMISSION_G'][has_redshift]
lrgs_chunk['FLUX_R_CORR'] = lrgs['FLUX_R'][has_redshift] / lrgs['MW_TRANSMISSION_R'][has_redshift]
lrgs_chunk['FLUX_Z_CORR'] = lrgs['FLUX_Z'][has_redshift] / lrgs['MW_TRANSMISSION_Z'][has_redshift]
lrgs_chunk['FLUX_W1_CORR'] = lrgs['FLUX_W1'][has_redshift] / lrgs['MW_TRANSMISSION_W1'][has_redshift]
lrgs_chunk['FLUX_W2_CORR'] = lrgs['FLUX_W2'][has_redshift] / lrgs['MW_TRANSMISSION_W2'][has_redshift]

lrgs_chunk['FLUX_SIG_G'] = lrgs['FLUX_IVAR_G'][has_redshift]**-0.5
lrgs_chunk['FLUX_SIG_R'] = lrgs['FLUX_IVAR_R'][has_redshift]**-0.5
lrgs_chunk['FLUX_SIG_Z'] = lrgs['FLUX_IVAR_Z'][has_redshift]**-0.5
lrgs_chunk['FLUX_SIG_W1'] = lrgs['FLUX_IVAR_W1'][has_redshift]**-0.5
lrgs_chunk['FLUX_SIG_W2'] = lrgs['FLUX_IVAR_W2'][has_redshift]**-0.5

# lrgs_chunk['SEDFLOW_SAMPLES'] = np.empty((np.sum(has_redshift), 100, 15)) # 100 subsamples from the posterior
# lrgs_chunk['SEDFLOW_MAP'] = np.empty((np.sum(has_redshift), 15)) # maximum a posteriori
# lrgs_chunk['SEDFLOW_LOGMSTAR_SAMPLES'] = np.empty((np.sum(has_redshift), 100))
# lrgs_chunk['SEDFLOW_LOGMSTAR_MAP'] = np.empty(np.sum(has_redshift))

/tmp/ipykernel_2147654/2426835379.py:13: RuntimeWarning: divide by zero encountered in power
  lrgs_chunk['FLUX_SIG_G'] = lrgs['FLUX_IVAR_G'][has_redshift]**-0.5


In [6]:
lrgs_chunk.write('/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1_lrg/LRG_full_Y1.iron_v1.2.fluxes.hdf5', 
                 overwrite=True)

/global/homes/c/chahah/.conda/envs/gqp/lib/python3.12/site-packages/astropy/io/misc/hdf5.py:281: UserWarning: table path was not set via the path= argument; using default path __astropy_table__
  warnings.warn(


below is an attemp to match lrg-act to lrg. this didn't work because the lrg-act target ids were rewritten to float and that changed the values. Not worth dealing with. 

In [10]:
n_post = 0
for igal, ra, dec, tid, zred in zip(np.arange(len(lrgs_act)), lrgs_act['col2'], lrgs_act['col3'], lrgs_act['col1'], lrgs_act['col4']): 
    i_min = np.argmin((lrgs_chunk['RA'] - ra)**2 + (lrgs_chunk['DEC'] - dec)**2)
    r_min = np.sqrt((lrgs_chunk['RA'][i_min] - ra)**2 + (lrgs_chunk['DEC'][i_min] - dec)**2)
    
    if not isinstance(lrgs_chunk['Z_not4clus'][i_min], float): continue 

    if np.isclose(r_min, 0., atol=1e-1) and np.isclose(lrgs_chunk['Z_not4clus'][i_min], zred, atol=0.01): 
        f_mc = '/global/cfs/projectdirs/desi/users/chahah/sedflow/desiy1_della/desiy1.lrg_act.sedflow.modelb.lowz.cdf.grzW1W2.%i.npy' % igal
        if os.path.isfile(f_mc): 
            lrgs_chunk['SEDFLOW_SAMPLES'][i_min] = np.load(f_mc)[::100]
                        
            # calculate log M* (surviving mass)
            tage = Planck13.age(lrgs_chunk['Z_not4clus'][i_min]).value # age in Gyr
            lrgs_chunk['SEDFLOW_LOGMSTAR_SAMPLES'][i_min] = gsed._msurv(lrgs_chunk['SEDFLOW_SAMPLES'][i_min], np.repeat(tage, 100))            
            
            print(int(tid), zred)
            print(lrgs_chunk['TARGETID'][i_min], lrgs_chunk['Z_not4clus'][i_min])
            n_post += 1

/global/homes/c/chahah/.conda/envs/gqp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/global/u1/c/chahah/projects/SEDflow/src/sedflow/galaxy.py:179: RuntimeWarning: invalid value encountered in log10
  thetas[:,6] = np.log10(thetas[:,6])
/global/u1/c/chahah/projects/SEDflow/src/sedflow/galaxy.py:180: RuntimeWarning: invalid value encountered in log10
  thetas[:,7] = np.log10(thetas[:,7])
/global/u1/c/chahah/projects/SEDflow/src/sedflow/galaxy.py:217: RuntimeWarning: invalid value encountered in log10
  thetas[:,6] = np.log10(thetas[:,6])
/global/u1/c/chahah/projects/SEDflow/src/sedflow/galaxy.py:218: RuntimeWarning: invalid value encountered in log10
  thetas[:,7] = np.log10(thetas[:,7])


39627398206985200 0.40002502690194286
39627398206985200 0.40002502690194286
39627485616283136 0.40005466884443336
39627485616283134 0.40005466884443336
39627461914264768 0.40011670412479533
39627461914264767 0.40011670412479533
39627398186010576 0.4001301468177446
39627398186010580 0.4001301468177446
39627479882663344 0.4001390278076434
39627479882663343 0.4001390278076434
39627484794195984 0.40017432764083777
39627484794195988 0.40017432764083777
39627369048181896 0.40032170883258217
39627369048181895 0.40032170883258217
39627509456708144 0.40036492580788785
39627509456708140 0.40036492580788785
39627503714698048 0.4004381568233545
39627503714698046 0.4004381568233545
39627375041841528 0.4004769667191899
39627375041841529 0.4004769667191899
39627502389301472 0.40055542499056823
39627502389301474 0.40055542499056823
39627444872811848 0.40064754785209983
39627444872811846 0.40064754785209983
39627357744531104 0.400655096367006
39627357744531100 0.400655096367006
39627467656266504 0.4006

EOFError: No data left in file